# Telco Churn Master Notebook v2.1

This notebook is a portfolio-grade Phase 0 to Phase 4 analysis for the Telco Customer Churn dataset.
It is organized by ways of thinking rather than isolated techniques:

Business Thinking -> Data Thinking -> Statistical Thinking -> Feature Thinking -> Information Thinking -> Feature Selection -> Model Readiness.

Source file: `data\WA_Fn-UseC_-Telco-Customer-Churn.csv`

Generated on: 2026-07-22 23:10:03


## tl;dr

### Business Summary
- The observed churn rate is **26.54%** across **7,043** customers.
- The highest-risk contract segment is **Month-to-month**, with a churn rate of **42.71%**.
- The lowest-risk contract segment is **Two year**, with a churn rate of **2.83%**.
- Churned customers have much shorter tenure: median tenure is **10 months** for churned customers versus **38 months** for retained customers.

### Technical Summary
- Strongest evidence appears around **contract commitment, tenure maturity, price pressure, support/protection gaps, and spending intensity**.
- Top information/model signals include `ChargesTenureRatio` by mutual information and `ChargesTenureRatio` by permutation importance.
- The final Phase 5 candidate feature set contains **20 semantic features**.

### Recommended Actions
- Prioritize retention for month-to-month customers, especially those with high monthly charges and weak support/protection attachment.
- Use engineered features such as `LoyaltyScore`, `SecurityScore`, `ChargesTenureRatio`, `PricePressureProxy`, and `SupportGapProxy` in Phase 5 modeling.
- Treat this notebook as the model-readiness handoff; the next step is baseline modeling, threshold tuning, and business-cost evaluation.

Important caveat: this notebook identifies predictive associations, not causal proof.


## Setup

The notebook uses the project virtual environment packages. If rerunning manually, make sure `pandas`, `numpy`, `scipy`, `sklearn`, and `plotly` are available.


In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats
from scipy.stats import chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, VarianceThreshold, chi2, mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df_raw = pd.read_csv(DATA_PATH)
df_raw.shape


(7043, 21)


## Chapter 1 - Business Thinking

### Goal

Understand the churn problem before touching models.

### Business Questions

- What is churn? A customer ending service with the company.
- Why do customers leave? Likely reasons include weak commitment, high price pressure, poor support coverage, insufficient perceived value, or low product attachment.
- What costs money? Lost recurring revenue, acquisition cost to replace the customer, and retention campaign spend.
- Which mistakes are expensive? Missing a truly risky high-value customer is expensive; over-targeting stable customers wastes retention budget.

### Stakeholder KPIs

- Churn rate
- Retention opportunity size
- Revenue-at-risk proxy
- Recall for churners
- Precision for retention campaigns
- Cost-adjusted campaign lift

### Potential Interventions

- Contract migration incentives for month-to-month customers.
- Support bundle offers for high-risk internet customers.
- Price review or loyalty discounts for high-charge, low-tenure customers.
- Onboarding focus during the first year of customer life.

### Question

- Why is churn prediction valuable?
- What is more expensive: false positives or false negatives?
- Why should we avoid causal language in this notebook?


## Chapter 2 - Data Thinking


In [2]:
data_quality_summary = pd.DataFrame([
    {"Check": "Rows", "Result": len(df_raw), "Decision": "Sufficient for portfolio-grade churn analysis"},
    {"Check": "Columns", "Result": df_raw.shape[1], "Decision": "Original schema retained before engineering"},
    {"Check": "Churn rate", "Result": (df_raw["Churn"].eq("Yes").mean()), "Decision": "Moderate class imbalance"},
    {"Check": "Duplicate customerID", "Result": df_raw["customerID"].duplicated().sum(), "Decision": "No customer-level duplicates found"},
    {"Check": "Blank TotalCharges", "Result": df_raw["TotalCharges"].astype(str).str.strip().eq("").sum(), "Decision": "Convert to numeric and flag"},
    {"Check": "Standard missing values", "Result": df_raw.isna().sum().sum(), "Decision": "No native NaN values found"},
])
data_quality_summary


Check,Result,Decision
Rows,7043,Sufficient for portfolio-grade churn analysis
Columns,21,Original schema retained before engineering
Churn rate,26.54%,"Moderate class imbalance; use ROC-AUC, PR-AUC, F1, recall"
Duplicate customerID,0,No customer-level duplicates found
Blank TotalCharges,11,Convert to numeric and flag; blanks are new/zero-tenure customers
Standard missing values,0,No native NaN values found


### Data Quality Interpretation

Observation: the dataset has no duplicate customer IDs and no native null values, but `TotalCharges` contains blank strings.

Business interpretation: the blank `TotalCharges` records likely represent new customers with no accumulated billing yet.

Engineering implication: convert `TotalCharges` to a numeric field and preserve a blank-value flag for auditability.

Takeaway: missingness can be hidden as strings, so schema checks alone are not enough.


In [3]:
feature_taxonomy


Feature,Taxonomy,UniqueValues,BusinessRole
customerID,Nominal categorical,7043,Identifier
gender,Binary,2,Predictor
SeniorCitizen,Binary,2,Predictor
Partner,Binary,2,Predictor
Dependents,Binary,2,Predictor
tenure,Numerical,73,Predictor
PhoneService,Binary,2,Predictor
MultipleLines,Nominal categorical,3,Predictor
InternetService,Nominal categorical,3,Predictor
OnlineSecurity,Nominal categorical,3,Predictor


In [4]:
churn_counts = df_raw["Churn"].value_counts().reset_index()
churn_counts.columns = ["Churn", "Customers"]
churn_counts["Share"] = churn_counts["Customers"] / len(df_raw)
fig = px.bar(churn_counts, x="Churn", y="Customers", text="Customers", title="Churn Distribution")
fig.show()


### Visualization Interpretation - Churn Distribution

Observation: **26.54%** of customers churned.

Business interpretation: churn is a major enough population to justify retention investment.

Engineering implication: the target is imbalanced but not extremely rare, so PR-AUC, recall, F1, and cost-aware thresholds matter.

Takeaway: accuracy alone would be misleading because always predicting "No churn" already performs strongly on accuracy.


In [ ]:
contract_churn = (
    df.groupby("Contract")
      .agg(Customers=("customerID", "count"), ChurnRate=("ChurnBinary", "mean"))
      .reset_index()
      .sort_values("ChurnRate", ascending=False)
)
fig = px.bar(contract_churn, x="Contract", y="ChurnRate", text="ChurnRate", title="Churn Rate by Contract Type")
fig.show()


Contract,Customers,ChurnRate,ChurnRatePct
Month-to-month,3875,0.4271,42.71
One year,1473,0.1127,11.27
Two year,1695,0.0283,2.83


### Visualization Interpretation - Contract vs Churn

Observation: month-to-month customers churn at **42.71%**, while two-year customers churn at **2.83%**.

Business interpretation: long-term commitment is one of the clearest retention signals.

Engineering implication: `Contract` should be treated as a high-priority predictive feature, and `ContractCommitmentScore` is a useful engineered representation.

Takeaway: contract can outperform raw billing features because it directly captures switching friction and customer commitment.


In [6]:
fig = px.box(df, x="Churn", y="tenure", color="Churn", title="Tenure Distribution by Churn Outcome")
fig.show()


Churn,Customers,MeanTenure,MedianTenure
No,5174,37.57,38.0
Yes,1869,17.98,10.0


### Visualization Interpretation - Tenure vs Churn

Observation: churned customers are concentrated at much lower tenure; median tenure is 10 months for churned customers and 38 months for retained customers.

Business interpretation: the first year is a critical retention window.

Engineering implication: tenure should remain available both as a raw numeric feature and as lifecycle-stage features.

takeaway: median is important here because high-tenure churners pull the mean upward.


In [7]:
fig = px.box(df, x="Churn", y="MonthlyCharges", color="Churn", title="Monthly Charges by Churn Outcome")
fig.show()


Churn,Customers,MeanMonthlyCharges,MedianMonthlyCharges
No,5174,61.27,64.43
Yes,1869,74.44,79.65


### Visualization Interpretation - MonthlyCharges vs Churn

Observation: churned customers have higher monthly charges on average.

Business interpretation: pricing pressure is a plausible churn driver, especially when combined with low commitment.

Engineering implication: monthly charge should be combined with tenure and contract to create price-pressure features.

Takeaway: raw `MonthlyCharges` is useful, but interactions such as high charge plus month-to-month contract can be more meaningful.


## Chapter 3 - Statistical Thinking


In [8]:
descriptive


Feature,Mean,Median,Std,Skewness,Kurtosis,CV,Q1,Q3,IQR,OutlierPct
tenure,32.3711,29.00,24.5595,0.2395,-1.3874,0.7587,9.00,55.00,46.00,0.0
MonthlyCharges,64.7617,70.35,30.0900,-0.2205,-1.2573,0.4646,35.50,89.85,54.35,0.0
TotalCharges_num,2279.7343,1394.55,2266.7945,0.9632,-0.2286,0.9943,398.55,3786.60,3388.05,0.0


### Descriptive Statistics Interpretation

Observation: tenure and total charges are skewed because many customers are early in their lifecycle while some stay for years.

Business interpretation: mean alone can hide early churn risk; median and IQR explain the typical customer more honestly.

Engineering implication: skewness supports ratio features, lifecycle bins, and robust summaries.

takeaway: descriptive statistics are not filler; they tell us which transformations and model assumptions deserve attention.


In [9]:
chi_square_results


Feature,ChiSquare,PValue,Dof,CramersV,BusinessDecision
Contract,1184.59657,0.0,2,0.41012,Strong association to investigate
OnlineSecurity,849.99897,0.0,2,0.34740,Strong association to investigate
TechSupport,828.19707,0.0,2,0.34292,Strong association to investigate
InternetService,732.30959,0.0,2,0.32245,Strong association to investigate
PaymentMethod,648.14233,0.0,3,0.30336,Strong association to investigate
PaperlessBilling,258.27765,0.0,1,0.19150,Strong association to investigate
Dependents,189.12925,0.0,1,0.16387,Strong association to investigate
SeniorCitizen,159.42630,0.0,1,0.15045,Strong association to investigate
Partner,158.73338,0.0,1,0.15013,Strong association to investigate


### Inferential Statistics - Categorical Features

Observation: contract, online security, tech support, internet service, and payment method show strong association with churn.

Business interpretation: churn is connected to commitment, product experience, and billing behavior.

Engineering implication: keep these features or carefully designed proxies in the candidate set.

takeaway: p-values answer whether an association is unlikely under independence; Cramer's V helps judge practical strength.


In [10]:
numeric_tests


Feature,Mean_Churned,Mean_Stayed,Median_Churned,Median_Stayed,WelchT_PValue,MannWhitney_PValue,CohensD_ChurnMinusStay
tenure,17.97913,37.56997,10.00,38.000,0.0,0.0,-0.85225
MonthlyCharges,74.44133,61.26512,79.65,64.425,0.0,0.0,0.44628
TotalCharges_num,1531.79609,2549.91144,703.55,1679.525,0.0,0.0,-0.45821


### Inferential Statistics - Numeric Features

Observation: tenure, monthly charges, and total charges differ meaningfully between churned and retained customers.

Business interpretation: churn is connected to lifecycle maturity and customer economics.

Engineering implication: numeric features should be retained, but redundancy among tenure, total charges, and CLV-like features must be controlled.

takeaway: use non-parametric tests such as Mann-Whitney when distributions are skewed and normality is questionable.


In [11]:
churn_rate_intervals


Segment,Customers,ChurnRate,CI95Low,CI95High
Contract=Month-to-month,3875,0.42710,0.41160,0.44274
Contract=One year,1473,0.11270,0.09754,0.12986
Contract=Two year,1695,0.02832,0.02143,0.03735


## Chapter 4 - Feature Thinking


In [12]:
feature_catalog


Feature,Category,BusinessConcept,FeatureType,Formula,Rationale,ExpectedChurnSignal
ServiceCount,Business,Service adoption,Engineered count,"sum(OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies == Yes)",More subscribed services may indicate deeper product adoption.,"Higher adoption should generally reduce churn, except for high-price segments."
SecurityScore,Business,Protection,Engineered count,OnlineSecurity + OnlineBackup + DeviceProtection + TechSupport,"Protection services represent security, backup, device, and support attachment.",Higher protection should reduce churn.
EntertainmentScore,Behavioral,Entertainment dependency,Engineered count,StreamingTV + StreamingMovies,Streaming services may represent lifestyle dependence on the telecom bundle.,Higher entertainment dependence may reduce churn if value is perceived.
ContractCommitmentScore,Business,Commitment,Ordinal encoding,"Month-to-month=0, One year=1, Two year=2",Longer contracts directly encode switching friction and commitment.,Higher commitment should strongly reduce churn.
TenureScore,Business,Relationship maturity,Binned ordinal,"0-11 months=0, 12-23 months=1, 24+ months=2",Early customer lifecycle is visibly higher risk than mature tenure.,Higher tenure maturity should reduce churn.
LoyaltyScore,Behavioral,Loyalty and commitment,Composite score,TenureScore + ContractCommitmentScore,Combines relationship duration and contractual commitment.,Higher loyalty should reduce churn.
ChargesTenureRatio,Mathematical,Spending intensity,Ratio,MonthlyCharges / (tenure + 1),Highlights customers paying a lot relative to relationship maturity.,Higher intensity can indicate price pressure among newer customers.
EstimatedCLV,Mathematical,Revenue contribution,Product,MonthlyCharges * tenure,Approximates accumulated value when actual historical revenue is unavailable.,Useful for prioritization but redundant with TotalCharges.
MonthToMonth_HighCharge,Interaction,Price pressure,Binary interaction,Contract == Month-to-month and MonthlyCharges >= Q3,Combines high bill size with low contractual commitment.,Should increase churn risk.
Fiber_NoTechSupport,Interaction,Support gap,Binary interaction,InternetService == Fiber optic and TechSupport == No,High-value service without support may create dissatisfaction.,Should increase churn risk.


### Feature Catalog Interpretation

Observation: engineered features are organized by business concept, not by random transformations.

Business interpretation: the notebook turns raw columns into concepts stakeholders understand: commitment, protection, price pressure, support gap, and digital lifestyle.

Engineering implication: the feature catalog becomes a reusable reference for modeling, review, and future production work.

takeaway: senior feature engineering starts with hidden business concepts, then translates them into measurable features.


In [13]:
fig = px.bar(service_churn, x="ServiceCount", y="ChurnRatePct", text="ChurnRatePct", title="Churn Rate by ServiceCount")
fig.show()


ServiceCount,Customers,ChurnRate,ChurnRatePct
0,2219,0.214060,21.41
1,966,0.457557,45.76
2,1033,0.358180,35.82
3,1118,0.273703,27.37
4,852,0.223005,22.30
5,571,0.124343,12.43
6,284,0.052817,5.28


### Visualization Interpretation - ServiceCount

Observation: service adoption is related to churn, but the pattern is not purely linear because non-internet customers form a different segment.

Business interpretation: product adoption matters, but we must compare similar customers.

Engineering implication: split broad service adoption into more meaningful groups such as `SecurityScore` and `EntertainmentScore`.

takeaway: a feature can be directionally useful but still need refinement because it mixes populations.


In [14]:
fig = px.bar(security_churn, x="SecurityScore", y="ChurnRatePct", text="ChurnRatePct", title="Churn Rate by SecurityScore")
fig.show()


SecurityScore,Customers,ChurnRate,ChurnRatePct
0,2793,0.297530,29.75
1,1467,0.388548,38.85
2,1372,0.237609,23.76
3,941,0.124336,12.43
4,470,0.053191,5.32


### Visualization Interpretation - SecurityScore

Observation: customers with more protection services churn less, especially at high protection scores.

Business interpretation: protection products appear to represent ecosystem attachment and risk aversion.

Engineering implication: `SecurityScore` is a strong stakeholder-friendly engineered feature.

takeaway: combining related columns can reduce dimensionality while improving interpretability.


In [15]:
fig = px.bar(loyalty_churn, x="LoyaltyScore", y="ChurnRatePct", text="ChurnRatePct", title="Churn Rate by LoyaltyScore")
fig.show()


LoyaltyScore,Customers,ChurnRate,ChurnRatePct
0,1908,0.519392,51.94
1,867,0.341407,34.14
2,1460,0.271918,27.19
3,1255,0.109163,10.92
4,1553,0.030908,3.09


### Visualization Interpretation - LoyaltyScore

Observation: churn declines sharply as the loyalty score increases.

Business interpretation: tenure maturity and contract commitment work together as a loyalty signal.

Engineering implication: `LoyaltyScore` is useful, but raw `tenure` and `Contract` should remain available for audit.

takeaway: composite features improve storytelling but can be redundant with their components.


In [16]:
risk_register


Feature,LeakageRisk,ProxyRisk,BiasRisk,DeploymentRisk,Mitigation
TotalCharges_num,Low if known at prediction time,Revenue and tenure proxy,Can overweight long-tenured customers,Requires consistent billing history availability,Validate that total charges are available before retention scoring.
EstimatedCLV,Low,Almost duplicates TotalCharges in this dataset,Can bias actions toward high-revenue customers,Needs consistent definition across billing systems,"Use for prioritization, not as a mandatory predictive input."
LoyaltyScore,Low,Combines tenure and contract,May undervalue new customers who would become loyal,Stable if tenure and contract are current,Keep raw components available for audit.
PricePressureProxy,Low,Uses payment behavior and charges as affordability proxies,May reflect socioeconomic patterns not directly observed,Thresholds can drift as pricing changes,Monitor segment fairness and recalibrate thresholds.
SupportGapProxy,Low,"Approximates unmet need, not actual support experience",May confuse product mix with service quality,Depends on accurate support-service status,Pair with actual support-ticket data in future versions.
DigitalLifestyleScore,Low,Approximates digital engagement from product adoption,May reflect customer demographics indirectly,Definitions change if product catalog changes,Version feature definition and monitor distribution drift.


### Leakage, Proxy, Bias, and Deployment Risk

Observation: most engineered features are available at prediction time, but several are proxy variables.

Business interpretation: proxies are useful, but they must be explained carefully before stakeholders act on them.

Engineering implication: track leakage, proxy risk, bias risk, and deployment risk as part of feature documentation.

takeaway: feature engineering is not only about lift; it is also about trust, fairness, and operational reliability.


## Chapter 5 - Information Thinking


In [17]:
parent_entropy = entropy_from_series(df["Churn"])
info_gain_table


Parent entropy: 0.8347 bits


Feature,InformationGain
LoyaltyScore,0.14262
Contract,0.14204
PricePressureProxy,0.14161
SupportGapProxy,0.08552
InternetService,0.08018
TenureBand_IG17,0.07502
PaymentMethod,0.06423
DigitalLifestyleScore,0.04161
ServiceCount,0.04125
SecurityScore,0.03922


### Information Thinking Interpretation

Observation: features such as contract, loyalty, price pressure, and support gap reduce uncertainty about churn.

Business interpretation: the model learns by receiving information that makes churn less surprising.

Engineering implication: information gain and mutual information help decide whether a feature deserves to exist.

takeaway: entropy is a confusion measure; information gain is the reduction in that confusion after seeing a feature.


In [18]:
fig = px.line(tenure_gain_scan, x="Threshold", y="InformationGain", markers=True, title="Information Gain for Every Tenure Threshold")
fig.show()


Best tenure split: tenure <= 17, information gain = 0.07502


### Visualization Interpretation - Tenure Threshold Scan

Observation: the best single tenure split is around **17 months**, with information gain **0.07502**.

Business interpretation: customers appear to stabilize after the early lifecycle period.

Engineering implication: tenure can be represented both continuously and as lifecycle bands.

takeaway: decision trees search thresholds by asking which split reduces uncertainty the most.


## Chapter 6 - Evidence-Based Feature Selection


In [19]:
low_variance_features.sort_values("Variance").head(15)


Feature,Variance,Keep
TotalCharges_was_blank,0.001560,False
MonthlyChargePercentile,0.083345,True
PhoneService_No,0.087469,True
PhoneService_Yes,0.087469,True
MultipleLines_No phone service,0.087469,True
RiskAversionProxy,0.103504,True
MonthToMonth_HighCharge,0.109031,True
SeniorCitizen,0.135875,True
Contract_One year,0.165426,True
PaymentMethod_Credit card (automatic),0.169425,True


### Variance Threshold

Question: does this feature vary enough to be useful?

Observation: very rare flags, including blank-charge indicators, carry little standalone variance.

Business interpretation: rare events may still matter operationally, but they rarely carry broad predictive signal alone.

Engineering implication: remove or keep rare flags only when there is a clear business reason.

takeaway: low variance is a filter method; it does not know the target.


In [20]:
high_correlation_pairs


FeatureA,FeatureB,Correlation
ServiceCount,InternetServiceCount,1.0000
TotalCharges_num,EstimatedCLV,0.9996
SecurityScore,InternetServiceCount,0.9133
ServiceCount,SecurityScore,0.9133
TenureScore,LoyaltyScore,0.8836
tenure,LoyaltyScore,0.8793
ContractCommitmentScore,LoyaltyScore,0.8680
tenure,TenureScore,0.8633
MonthlyCharges,DigitalLifestyleScore,0.8535


In [21]:
fig = px.imshow(corr_matrix.round(2), text_auto=True, aspect="auto", color_continuous_scale="RdBu", zmin=-1, zmax=1, title="Correlation Heatmap")
fig.show()


### Correlation Interpretation

Observation: `TotalCharges_num` and `EstimatedCLV` are nearly identical, and service count variants are highly redundant.

Business interpretation: multiple features may be telling the same business story.

Engineering implication: avoid keeping all redundant versions in linear models; choose the clearest representation.

takeaway: correlated features are not always bad for tree models, but they can destabilize linear coefficients and explanations.


In [22]:
vif_table


Feature,VIF
ServiceCount,inf
SecurityScore,inf
EntertainmentScore,inf
EstimatedCLV,1149.846554
TotalCharges_num,1139.502372
LoyaltyScore,14.717536
tenure,12.294833
MonthlyCharges,8.477947
DigitalLifestyleScore,7.877252
ContractCommitmentScore,7.030446


### VIF Interpretation

Observation: exact and near-exact engineered combinations produce very high or infinite VIF.

Business interpretation: composite features must be useful enough to justify redundancy.

Engineering implication: for linear models, prefer either raw components or composite scores, not every duplicate representation.

takeaway: VIF answers a different question than predictive importance; it detects multicollinearity among predictors.


In [23]:
mutual_information.head(15)


Feature,MutualInformation
ChargesTenureRatio,0.13264
ContractCommitmentScore,0.10445
LoyaltyScore,0.09865
PricePressureProxy,0.09830
Contract_Month-to-month,0.09297
tenure,0.07229
TenureBand_IG17,0.05971
Fiber_NoTechSupport,0.05961
OnlineSecurity_No,0.05876
TechSupport_No,0.05665


In [24]:
chi2_encoded_table.head(15)


Feature,Chi2,PValue
EstimatedCLV,624391.17674,0.0
TotalCharges_num,624292.00300,0.0
tenure,16278.92369,0.0
ChargesTenureRatio,15740.77914,0.0
MonthlyCharges,3680.78770,0.0
AvgChargePerTenureMonth,3673.57999,0.0
LoyaltyScore,1431.02785,0.0
ContractCommitmentScore,1115.78017,0.0
PricePressureProxy,891.75737,0.0
Fiber_NoTechSupport,596.47815,0.0


### Mutual Information and Chi-square

Observation: commitment, tenure, charges, support, and internet-service signals repeatedly appear near the top.

Business interpretation: churn risk is not random; it clusters around commitment, product value, and support coverage.

Engineering implication: keep features that repeatedly show signal across independent selection methods.

takeaway: mutual information captures general dependence, while chi-square is useful for categorical association.


In [25]:
l1_coefficients.head(15)


Feature,AbsCoefficient
MonthlyCharges,0.71872
DigitalLifestyleScore,0.52650
ContractCommitmentScore,0.51166
tenure,0.48361
InternetService_Fiber optic,0.40883
ChargesTenureRatio,0.39789
TotalCharges_num,0.23567
TotalCharges_was_blank,0.19251
LoyaltyScore,0.18807
OnlineSecurity_No,0.14579


In [26]:
rfe_table.head(20)


Feature,Selected,Rank
MultipleLines_Yes,True,1
InternetService_DSL,True,1
InternetService_Fiber optic,True,1
TechSupport_No internet service,True,1
StreamingTV_No internet service,True,1
StreamingMovies_No internet service,True,1
tenure,True,1
MonthlyCharges,True,1
TotalCharges_num,True,1
TotalCharges_was_blank,True,1


### LASSO and RFE

Observation: optimization-based and wrapper-based methods retain several of the same business drivers.

Business interpretation: the engineered concepts are not only explainable; several are also useful to models.

Engineering implication: compare feature sets by validation performance rather than trusting one selector blindly.

takeaway: LASSO selects by shrinking coefficients; RFE selects by repeatedly fitting a model and removing weaker predictors.


In [27]:
tree_importance.head(15)


Feature,TreeImportance
ChargesTenureRatio,0.12963
Contract_Month-to-month,0.11313
ContractCommitmentScore,0.10446
LoyaltyScore,0.08083
PricePressureProxy,0.06530
OnlineSecurity_No,0.04046
Fiber_NoTechSupport,0.03668
tenure,0.03588
TechSupport_No,0.03369
Contract_Two year,0.03073


In [28]:
permutation_importance_table.head(15)


Feature,PermutationImportance
ChargesTenureRatio,0.01086
Contract_Month-to-month,0.00767
ContractCommitmentScore,0.00697
LoyaltyScore,0.00549
PricePressureProxy,0.00375
tenure,0.00284
Contract_Two year,0.00242
TenureBand_IG17,0.00200
InternetService_Fiber optic,0.00184
Fiber_NoTechSupport,0.00178


In [29]:
boruta_style_table.head(20)


Feature,Importance,MaxShadowImportance,BetterThanShadow
ChargesTenureRatio,0.105580,0.001969,True
ContractCommitmentScore,0.098051,0.001969,True
LoyaltyScore,0.080571,0.001969,True
Contract_Month-to-month,0.075115,0.001969,True
PricePressureProxy,0.060513,0.001969,True
OnlineSecurity_No,0.044078,0.001969,True
tenure,0.042913,0.001969,True
SupportGapProxy,0.037328,0.001969,True
TechSupport_No,0.029247,0.001969,True
Contract_Two year,0.028814,0.001969,True


### Tree, Permutation, Boruta-style, and Optional SHAP

Observation: tree and permutation methods again emphasize commitment, spending intensity, support, and internet-service signals.

Business interpretation: these are stable enough to carry into baseline modeling.

Engineering implication: Boruta-style shadow features provide a useful sanity check against noise without installing another package.

SHAP status: SHAP is not installed in this environment, so this notebook records the method as optional rather than adding an unvalidated dependency.

takeaway: tree importance measures split usage, permutation importance measures performance drop, and SHAP explains contribution if the package is available and validated.


In [30]:
subset_performance


Subset,Features,ROC_AUC,PR_AUC,F1_at_0_5
All encoded features,64,0.84983,0.66352,0.62116
Top 15 mutual information,15,0.84535,0.65359,0.62679
Top 15 permutation,15,0.84517,0.65259,0.62881
RFE selected 15,15,0.84732,0.65819,0.61695
L1 selected,40,0.85001,0.66418,0.61893


In [31]:
feature_decision_matrix


Feature,BusinessValue,StatisticalEvidence,ModelEvidence,LeakageRisk,Interpretability,FinalDecision
customerID,Identifier only,No meaningful population signal,Excluded before encoding,Memorization risk,Low,Drop
TotalCharges,Historical billed amount,11 blanks; converted to TotalCharges_num,Raw string unsuitable,Low if available at scoring time,Medium,Drop raw; keep numeric version
Contract,Commitment and switching friction,Churn ranges from 2.83% to 42.71% by contract,MI:0.0930; MI:0.0511; Tree:0.1131; Tree:0.0307; Permutation:0.0077; Permutation:0.0024; LASSO:0.0662,Low,High,Keep
tenure,Relationship maturity,Churned median tenure 10 months vs stayed 38,MI:0.0723; Tree:0.0359; Permutation:0.0028; LASSO:0.4836,Low,High,Keep
MonthlyCharges,Current bill size,Higher average charges among churned customers,MI:0.0462; Tree:0.0165; Permutation:0.0004; LASSO:0.7187,Low,High,Keep
TotalCharges_num,Accumulated billed value,Strong numerical difference but highly correlated with tenure and EstimatedCLV,MI:0.0447; Tree:0.0193; Permutation:0.0016; LASSO:0.2357; LASSO:0.1925,Low if available before scoring,Medium,Keep with redundancy monitoring
EstimatedCLV,Revenue prioritization proxy,Correlation nearly identical to TotalCharges_num,"High chi-square due scale, but redundant",Low,Medium,Drop from core model; use for prioritization
ServiceCount,Overall service adoption,Perfectly redundant with InternetServiceCount in this dataset,Lower priority after split into SecurityScore and EntertainmentScore,Low,High,Candidate; prefer component scores
SecurityScore,Protection and risk aversion,Churn falls to 5.32% at max protection score,Strong business feature; component services also useful,Low,High,Keep
EntertainmentScore,Entertainment dependency,Useful behavioral grouping; weaker than commitment/protection,Selected by LASSO/RFE in dry run,Low,High,Keep


### Final Feature Decision Matrix

Observation: the final decision balances business value, statistical evidence, model evidence, leakage risk, and interpretability.

Business interpretation: stakeholders get a transparent reason for each keep/drop/candidate decision.

Engineering implication: this table is the handoff between feature selection and Phase 5 modeling.

takeaway: the best feature set is not simply the highest ranked list; it is a defensible set of useful, non-leaky, explainable predictors.


## Chapter 7 - Model Readiness


In [32]:
selected_features_df


Feature,Use
SeniorCitizen,Recommended Phase 5 candidate
Partner,Recommended Phase 5 candidate
Dependents,Recommended Phase 5 candidate
tenure,Recommended Phase 5 candidate
InternetService,Recommended Phase 5 candidate
OnlineSecurity,Recommended Phase 5 candidate
TechSupport,Recommended Phase 5 candidate
Contract,Recommended Phase 5 candidate
PaperlessBilling,Recommended Phase 5 candidate
PaymentMethod,Recommended Phase 5 candidate


### Readiness Report

Dataset status: cleaned and ready for baseline modeling.

Feature status: final candidate set contains **20** semantic features.

Train/test strategy: use stratified train/test split, then cross-validation after the baseline.

Baseline strategy: logistic regression and shallow decision tree before Random Forest, XGBoost, LightGBM, and CatBoost.

Evaluation strategy: track ROC-AUC, PR-AUC, F1, recall, precision, calibration, and cost-sensitive threshold behavior.

Best lightweight validation subset in this notebook: **L1 selected**, ROC-AUC **0.8500**, PR-AUC **0.6642**, F1 **0.6189**.

takeaway: model readiness means the dataset, feature logic, leakage checks, split strategy, and business metric are ready before advanced modeling begins.


## Chapter 8 - Knowledge Graph Thinking


In [33]:
knowledge_graph_nodes.head(20)


id,label,type
Telco Churn,Telco Churn,Business Problem
Commitment,Commitment,Business Concept
Contract,Contract,Original Feature
tenure,tenure,Original Feature
ContractCommitmentScore,ContractCommitmentScore,Engineered Feature
TenureScore,TenureScore,Engineered Feature
LoyaltyScore,LoyaltyScore,Engineered Feature
Protection,Protection,Business Concept
OnlineSecurity,OnlineSecurity,Original Feature
OnlineBackup,OnlineBackup,Original Feature


In [34]:
knowledge_graph_edges.head(30)


source,target,relationship
Telco Churn,Commitment,has_concept
Commitment,Contract,represented_by
Commitment,tenure,represented_by
Commitment,ContractCommitmentScore,represented_by
Commitment,TenureScore,represented_by
Commitment,LoyaltyScore,represented_by
Telco Churn,Protection,has_concept
Protection,OnlineSecurity,represented_by
Protection,OnlineBackup,represented_by
Protection,DeviceProtection,represented_by


### Knowledge Graph Interpretation

Observation: raw columns and engineered features can be organized under business concepts such as Commitment, Protection, Price Pressure, Digital Lifestyle, Support Gap, and Risk Aversion.

Business interpretation: this turns the dataset into a business ontology, not just a table.

Engineering implication: the exported nodes and edges can support your future Knowledge Graph and LLM/RAG portfolio work.

takeaway: knowledge graphs help make feature logic explainable, reusable, and connected to business meaning.


## Chapter 9 - Mastery

### Business Thinking
- Why is churn prediction valuable?
- Which mistake is more expensive: false positive or false negative?
- Why should we avoid causal claims here?

### Data Thinking
- Why is `TotalCharges` dangerous if loaded as a string?
- Why is target imbalance important?
- Why do we remove `customerID`?

### Statistical Thinking
- Why use median in addition to mean?
- Why might Mann-Whitney be safer than a t-test here?
- What does Cramer's V tell us that a p-value does not?

### Feature Thinking
- Why create `SecurityScore`?
- Why can `LoyaltyScore` be both useful and redundant?
- What is the difference between a business feature and a mathematical feature?

### Information Thinking
- What does entropy measure?
- Why does information gain matter for decision trees?
- Why is mutual information useful for feature selection?

### Feature Selection
- When should correlated features be removed?
- How is permutation importance different from tree importance?
- What does a Boruta-style shadow feature test protect against?

### Executive Presentation Exercise
Explain this notebook to a product leader in three sentences:

1. Churn is concentrated among low-commitment, early-tenure, high-price-pressure customers.
2. The strongest feature evidence points to commitment, tenure maturity, support/protection attachment, and billing behavior.
3. The dataset is ready for Phase 5 baseline modeling, but final business action requires threshold tuning and cost-sensitive evaluation.
